# Multi-Armed Bandits: Epsilon-Greedy, UCB & Thompson Sampling
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajit-ai/Data_Science/blob/main/10_Reinforcement_Learning/multi_armed_bandits.ipynb)

Bandits are RL with no state: pick an arm, observe reward, learn which machine pays. The explore/exploit tradeoff in its purest form - and the algorithm behind A/B testing, ad selection and menu optimization.

We race three classic strategies on the same slot machines and measure REGRET (reward lost vs always picking the best arm). Pure NumPy - runs instantly.

## 1. The casino

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
true_probs = [0.30, 0.55, 0.72]            # 3 arms, unknown to agents
best = max(true_probs)

def pull(arm):
    return float(rng.random() < true_probs[arm])

## 2. Three strategies

In [ ]:
class EpsilonGreedy:
    def __init__(self, n, eps):
        self.eps = eps
        self.counts = np.zeros(n); self.values = np.zeros(n)
    def act(self):
        if rng.random() < self.eps:
            return rng.integers(len(self.counts))
        return int(np.argmax(self.values))
    def update(self, a, r):
        self.counts[a] += 1
        self.values[a] += (r - self.values[a]) / self.counts[a]

class UCB1:
    def __init__(self, n, c=2.0):
        self.c = c; self.counts = np.zeros(n); self.values = np.zeros(n); self.t = 0
    def act(self):
        if self.t < len(self.counts):
            return self.t                       # play each arm once
        bonus = self.c * np.sqrt(np.log(self.t) / self.counts)
        return int(np.argmax(self.values + bonus))
    def update(self, a, r):
        self.t += 1
        self.counts[a] += 1
        self.values[a] += (r - self.values[a]) / self.counts[a]

class Thompson:
    def __init__(self, n):
        self.alpha = np.ones(n); self.beta = np.ones(n)
    def act(self):
        return int(np.argmax(rng.beta(self.alpha, self.beta)))
    def update(self, a, r):
        self.alpha[a] += r
        self.beta[a] += 1 - r

- **Epsilon-greedy**: explores randomly forever (eps fraction).
- **UCB1**: optimism - prefers arms with high value OR high uncertainty.
- **Thompson**: keeps a Beta posterior per arm; samples from posteriors = natural exploration decay.

## 3. The race (200 simulations x 3000 pulls)

In [ ]:
agents = {
    "eps-greedy .05": lambda: EpsilonGreedy(3, 0.05),
    "eps-greedy .20": lambda: EpsilonGreedy(3, 0.20),
    "UCB1":           lambda: UCB1(3),
    "Thompson":       lambda: Thompson(3),
}
HORIZON, SIMS = 3000, 200
regret_curves = {}

for name, factory in agents.items():
    total_regret = np.zeros(HORIZON)
    for _ in range(SIMS):
        agent = factory()
        for t in range(HORIZON):
            a = agent.act()
            r = pull(a)
            agent.update(a, r)
            total_regret[t] += best - true_probs[a]
    regret_curves[name] = np.cumsum(total_regret / SIMS)

plt.figure(figsize=(9, 5))
for name, curve in regret_curves.items():
    plt.plot(curve, label=name)
plt.xlabel("pull"); plt.ylabel("cumulative regret (avg)")
plt.title("Lower = smarter exploring"); plt.legend(); plt.show()

## 4. What Thompson learned about each arm

In [ ]:
agent = Thompson(3)
for _ in range(3000):
    a = agent.act(); agent.update(a, pull(a))

fig, axes = plt.subplots(1, 3, figsize=(13, 3))
xs = np.linspace(0, 1, 200)
from scipy import stats as sps
for ax, a in zip(axes, range(3)):
    pdf = sps.beta.pdf(xs, agent.alpha[a], agent.beta[a])
    ax.plot(xs, pdf)
    ax.axvline(true_probs[a], ls="--", c="r")
    ax.set_title(f"arm {a}: belief vs truth {true_probs[a]}")
plt.tight_layout(); plt.show()

print("pulls per arm:", agent.alpha.astype(int) - 1)

## Takeaways
- Fixed-eps agents never stop paying exploration tax; UCB/Thompson spend early then converge.
- Thompson sampling dominates in practice and is trivially Bayesian-updatable - default choice for adaptive A/B tests.
- Bandit -> full RL bridge: add STATES and transitions and you have Q-learning (`q_learning_intro.ipynb`).
- Contextual bandits (features per user) sit between the two - the real ad-tech workhorse.